# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step template for loading, exploring, and analyzing the FAIR² mass spectrometry dataset using the `mlcroissant` library. All references to record sets, fields, and columns use the entity `@id` as defined in the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Display the name and description from the metadata
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
# Print more general metadata
print(f"Schema version: {dataset.metadata.conformsTo}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")


## 2. Data Overview

Review available record sets, fields, and their `@id`s (identifiers).

*We use `.metadata.record_sets`, `.fields`, and `.columns` to systematically inspect the schema definition.*

In [ ]:
# List all record sets and their fields with @id values
print("=== Record Sets ===")
for rs in dataset.metadata.record_sets:
    print(f"Record Set: {rs.name}")
    print(f" - @id: {rs.id}")
    print(f" - Description: {rs.description}")
    print(f" - Fields:")
    for field in rs.fields:
        print(f"    * {field.name} (@id: {field.id}, data_type: {field.data_type})")
    print("-" * 40)

# Example: show first several records as dicts for one record set
if dataset.metadata.record_sets:
    overview_rs_id = dataset.metadata.record_sets[0].id
    print(f"\nFirst 2 records for record set '@id': {overview_rs_id}")
    for i, record in enumerate(dataset.records(record_set=overview_rs_id)):
        print(record)
        if i >= 1:
            break

## 3. Data Extraction

Load data from one or more record sets into pandas DataFrames for analysis. Use the record set and field `@id`s found above.

*We'll extract data tables from each available record set, with record sets and fields referenced only by their `@id` fields.*

In [ ]:
# Collect all record set @id's
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
print(f"Record sets found: {record_set_ids}\n")
for record_set_id in record_set_ids:
    # Get records as list of dicts
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set {record_set_id} with columns: {df.columns.tolist()}")

# For demonstration, display the first 5 rows of the main data table (first record_set)
main_rs_id = record_set_ids[0] if record_set_ids else None
if main_rs_id:
    print(f"\nFirst 5 rows of record set '{main_rs_id}':")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps:
- Filtering records based on specific criteria
- Normalizing numeric fields
- Grouping and aggregating data

*All fields are referenced by their `@id` as defined in the Croissant schema.*

In [ ]:
# --- EDA SETUP --- #
# Select a numeric field and grouping field
# Reference them by their @id's (choose typical protein quantification columns)
import numpy as np

# Step 1: Find a numeric field in the main record set
main_rs = dataset.metadata.record_sets[0] if dataset.metadata.record_sets else None
if main_rs:
    numeric_field = None
    group_field = None
    for field in main_rs.fields:
        if numeric_field is None and field.data_type in ['schema:Number', 'schema:Float', 'schema:Integer']:
            numeric_field = field.id
        if group_field is None and field.data_type == 'schema:Text':
            group_field = field.id

    print(f"Selected numeric field: {numeric_field}")
    print(f"Selected grouping field: {group_field}\n")

    df = dataframes[main_rs.id]

    # Ensure field types are numeric
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    # Filtering example: keep only records with positive values
    threshold = np.nanquantile(df[numeric_field], 0.25)
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (first 5 rows):")
    display(filtered_df.head())

    # Normalization: z-score of the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()

    print(f"\nFirst 5 normalized {numeric_field} values:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouped analysis (if grouping field exists)
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].agg(['mean', 'count', 'std']).sort_values('mean', ascending=False)
        print(f"\nSummary statistics grouped by {group_field} for {numeric_field}:")
        display(grouped_df.head())
else:
    print("No record set or numeric field found for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

Below we plot the histogram for the selected numeric field and a boxplot grouped by the categorical field, referencing all fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt

if main_rs and numeric_field and not filtered_df.empty:
    plt.figure(figsize=(7, 4))
    plt.hist(filtered_df[numeric_field].dropna(), bins=30, alpha=0.7)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If grouping field available and not too many unique values
    if group_field and filtered_df[group_field].nunique() <= 15:
        plt.figure(figsize=(10,5))
        filtered_df.boxplot(column=numeric_field, by=group_field, grid=False, rot=75)
        plt.title(f'{numeric_field} grouped by {group_field}')
        plt.suptitle('')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No data available for plotting.")

## 6. Conclusion

In this notebook, we demonstrated how to use the `mlcroissant` library to:

- Load a FAIR⁲ Croissant dataset schema directly via its URL
- Review dataset structure: record sets and fields, always using their `@id`
- Extract and analyze the dataset with pandas DataFrames
- Conduct basic exploratory data analysis and visualize results

All access to metadata, records, and fields was done via their declared Croissant `@id`. This approach ensures schema-robust processing, easy migration to different Croissant datasets, and compatibility with FAIR data pipelines.